In [ ]:
#CELL 1
import pandas
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re

In [ ]:
#CELL 2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
#CELL 3
def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

# sequence = read_file("warandpeace.txt")

In [ ]:
#CELL 4
sequence = "abcdefghijklmnopqrstuvwxyz" * 100

In [ ]:
#CELL 5
vocab = sorted(set(sequence))
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

# Convert the entire text-based data into numerical data
data = [char_to_idx[char] for char in sequence]

# Print mappings and data for verification
print(f"Vocabulary: {vocab}")
print(f"Character to Index Mapping: {char_to_idx}")
print(f"Index to Character Mapping: {idx_to_char}")
print(f"Sample Numerical Data: {data[:100]}")  # Show a sample of the numerical data

Vocabulary: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Character to Index Mapping: {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25}
Index to Character Mapping: {0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}
Sample Numerical Data: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,

In [ ]:
#CELL 6

'''class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []

        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target'''

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.sequences = []
        self.targets = []

        # Create sequences and targets
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

        print(f"Generated {len(self.sequences)} sequences of length {sequence_length}.")
        print(f"Sequence length: {len(self.sequences[0]) if self.sequences else 0}")
        print(f"Target length: {len(self.targets[0]) if self.targets else 0}")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target

In [ ]:
#CELL 7

# Setting hyperparameters
sequence_length = 300   # Length of each input sequence
stride = 50             # Stride for creating sequences
embedding_dim = 50       # Dimension of character embeddings
hidden_size = 128         # Number of features in the hidden state of the RNN
num_layers = 2          # Number of layers in the RNN
learning_rate = 0.001   # Learning rate for the optimizer
num_epochs = 15         # Number of epochs to train
batch_size = 8         # Batch size for training
vocab_size = len(vocab) # Vocabulary size
input_size = len(vocab) # Input size (same as vocab size for character-level RNN)
output_size = len(vocab) # Output size (same as vocab size for character-level RNN)

After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below

The **`sequence_length`** defines the length of each input sequence processed by the model, allowing it to capture patterns over a fixed context of characters. The **`stride`** determines the step size for creating overlapping sequences, balancing the trade-off between generating more training samples and introducing redundancy in the data. The **`embedding_dim`** specifies the size of the vector space used to represent each character, enabling the model to learn meaningful representations of characters in a lower-dimensional space. The **`hidden_size`** dictates the number of features in the RNN's hidden state, which directly impacts the model's capacity to learn complex patterns and dependencies in the data. The **`num_layers`** sets the number of stacked RNN layers, allowing the model to capture hierarchical features and deeper patterns in the input sequences. The **`learning_rate`** controls the speed of weight updates during training, influencing convergence and stability. The **`num_epochs`** specifies the total number of times the model passes through the entire dataset, while the **`batch_size`** determines the number of samples processed together in one forward and backward pass, affecting training speed and memory usage. Lastly, the **`vocab_size`** represents the total number of unique characters in the dataset, defining the input and output dimensions of the model.

In [ ]:
#CELL 8
# Convert the data into a PyTorch tensor
data_tensor = torch.tensor(data, dtype=torch.long)

# Calculate the training size (90% of the total data)
train_size = int(0.9 * len(data_tensor))

# Split the data into training and testing sets
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]

# Print the sizes to verify the split
print(f"Total data size: {len(data_tensor)}")
print(f"Training data size: {len(train_data)}")
print(f"Testing data size: {len(test_data)}")

Total data size: 2600
Training data size: 2340
Testing data size: 260


In [ ]:
#CELL 9
# Create training and testing datasets
# Ensure sequence length fits the dataset
if len(test_data) < sequence_length + 1:
    print("Adjusting sequence_length to fit test_data")
    sequence_length = len(test_data) - 1

# Create datasets
train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

print(f"Total batches in testing data: {len(test_loader)}")

Adjusting sequence_length to fit test_data
Generated 42 sequences of length 259.
Sequence length: 259
Target length: 259
Generated 1 sequences of length 259.
Sequence length: 259
Target length: 259
Train dataset size: 42
Test dataset size: 1
Total batches in testing data: 1


In [ ]:
#CELL 10
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

        # Fully connected layer to project hidden states to vocabulary size
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v]
        return logits, final_hidden

    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)

In [ ]:
#CELL 11
# Initialize your RNN model
model = CharRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size, embedding_dim=embedding_dim).to(device)

# Define the loss function (use Cross Entropy Loss)
criterion = nn.CrossEntropyLoss()

# Initialize the optimizer (use Adam optimizer with learning rate)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Print confirmation
print(f"Model initialized: {model}")
print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: Adam with learning rate = {learning_rate}")

Model initialized: CharRNN(
  (embedding): Embedding(26, 50)
  (fc): Linear(in_features=128, out_features=26, bias=True)
)
Loss function: CrossEntropyLoss
Optimizer: Adam with learning rate = 0.001


In [ ]:
#CELL 12
for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/15:   0%|          | 0/5 [00:00<?, ?it/s]<ipython-input-87-9819b9d307d2>:46: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
<ipython-input-87-9819b9d307d2>:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/15: 100%|██████████| 5/5 [00:00<00:00,  9.94it/s]


Epoch [1/15], Loss: 3.1362, Accuracy: 53.81%


Epoch 2/15: 100%|██████████| 5/5 [00:00<00:00,  9.77it/s]


Epoch [2/15], Loss: 2.7814, Accuracy: 99.22%


Epoch 3/15: 100%|██████████| 5/5 [00:00<00:00, 10.08it/s]


...

Epoch [13/15], Loss: 0.0295, Accuracy: 99.68%


Epoch 14/15: 100%|██████████| 5/5 [00:00<00:00,  9.35it/s]


Epoch [14/15], Loss: 0.0261, Accuracy: 99.67%


Epoch 15/15: 100%|██████████| 5/5 [00:00<00:00,  9.97it/s]

Epoch [15/15], Loss: 0.0251, Accuracy: 99.65%


In [ ]:
sequence = read_file('warandpeace.txt')

In [ ]:
with torch.no_grad():
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    for batch_inputs, batch_targets in tqdm(test_loader, desc="Testing"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        # Adjust hidden state for current batch size
        hidden = model.init_hidden(batch_inputs.size(0))

        # Forward pass
        output, hidden = model(batch_inputs, hidden)

        # Debug shapes
        print(f"Output shape: {output.shape}")  # [batch_size, sequence_length, vocab_size]
        print(f"Targets shape: {batch_targets.shape}")  # [batch_size, sequence_length]

        # Compute loss
        loss = criterion(output.view(-1, vocab_size), batch_targets.view(-1))
        total_loss += loss.item()

        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)
        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.numel()

    # Calculate average loss and accuracy
    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100

    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

Testing:   0%|          | 0/1 [00:00<?, ?it/s]<ipython-input-87-9819b9d307d2>:46: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
<ipython-input-87-9819b9d307d2>:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Testing: 100%|██████████| 1/1 [00:00<00:00, 51.31it/s]

Output shape: torch.Size([1, 259, 26])
Targets shape: torch.Size([1, 259])
Test Loss: 0.0146, Test Accuracy: 100.00%


In [96]:
def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)

    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
    Generates text using the trained RNN model character-by-character.
    """
    model.eval()  # Set the model to evaluation mode
    start_text = start_text.lower()  # Convert input text to lowercase
    generated_text = start_text  # Initialize the generated text with the start text

    # Check if all characters in start_text are in the vocabulary
    for char in start_text:
        if char not in char_to_idx:
            raise ValueError(f"Character '{char}' not in vocabulary!")

    # Convert start_text to indices
    input_sequence = [char_to_idx[char] for char in start_text]
    input_sequence = torch.tensor(input_sequence, dtype=torch.long).unsqueeze(0).to(device)

    hidden = model.init_hidden(1)  # Initialize hidden state for batch size 1

    with torch.no_grad():
        for _ in range(k):  # Generate `k` characters
            # Forward pass
            output, hidden = model(input_sequence, hidden)

            # Get the last output logits
            logits = output[:, -1, :]  # Shape: [1, vocab_size]

            # Sample the next character index
            next_char_idx = sample_from_output(logits, temperature).item()

            # Append the predicted character to the generated text
            generated_text += idx_to_char[next_char_idx]

            # Update input_sequence for the next prediction
            input_sequence = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)

    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")

    if start_text.lower() == 'exit':
        print("Exiting...")
        break

    n = len(start_text)
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0

    completed_text = generate_text(model, start_text, n, k, temperature)

    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: abcdefghij
Generated text: defghijklmnopqrst
Generated text: fghijklmnopqrs
Generated text: mnop
Enter the initial text (n characters, or 'exit' to quit): exit
Exiting...
